# [MAIN QUEST 01] Spaceship Titanic EDA & Feature Engineering

## 1. 프로젝트 개요
* **목적:** 2912년 우주선 'Spaceship Titanic'의 시공간 이상 현상 충돌 사건 데이터를 분석하고, 승객의 '차원 이동(Transported)' 여부를 예측합니다.
* **핵심 전략:** 데이터의 분포와 변수 간 상관관계를 시각화(EDA)하여 논리적인 전처리 근거를 수립하고, 최종적으로 앙상블 모델을 통해 성능을 극대화합니다.

## 2. 탐색적 데이터 분석(EDA) 및 논리적 전처리 근거

앞선 데이터 시각화와 분석을 통해 다음과 같은 핵심 인사이트를 얻었고, 이를 전처리에 반영했습니다.

1. **CryoSleep(동면)의 결정적 영향:** 동면 중인 승객은 차원 이동될 확률이 압도적으로 높습니다. 따라서 우주선 내 시설 지출액(`TotalSpend`)이 0원인 승객은 동면 상태였을 것이라는 도메인 지식을 활용해 결측치를 채웠습니다.
2. **객실(Cabin) 정보 활용:** 객실의 층(`Deck`)과 출발 행성(`HomePlanet`) 간의 뚜렷한 연관성을 발견하여, A/B/C/T 구역은 Europa로, G 구역은 Earth로 결측치를 복구했습니다.
3. **동승자 여부:** 혼자 탑승했는지(`Is_Alone`) 파생 변수를 추가하여 생존 패턴을 모델이 더 잘 학습하도록 도왔습니다.

In [27]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 통합 전처리 (Train/Test 한 번에 처리하여 정보 손실 방지)
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
passengers = test['PassengerId']

# Target 임시 분리 및 데이터 병합
y = train['Transported'].astype(float)
df = pd.concat([train.drop('Transported', axis=1), test], axis=0).reset_index(drop=True)

# 2. 극강의 피처 엔지니어링
spend_feats = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df['TotalSpend'] = df[spend_feats].sum(axis=1)

# Cabin 분리 (결측치는 임의의 값 Z로 묶어서 하나의 그룹으로 처리)
df['Cabin'] = df['Cabin'].fillna('Z/9999/Z')
df['Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
df['Num'] = df['Cabin'].apply(lambda x: int(x.split('/')[1]))
df['Side'] = df['Cabin'].apply(lambda x: x.split('/')[2])

# GroupSize 및 Family 특징 추출
df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
df['GroupSize'] = df['Group'].map(df['Group'].value_counts())
df['Is_Alone'] = (df['GroupSize'] == 1).astype(int)

# 3. 논리적 결측치 대치술 (도메인 지식 활용)
df.loc[(df['HomePlanet'].isnull()) & (df['Deck'].isin(['A', 'B', 'C', 'T'])), 'HomePlanet'] = 'Europa'
df.loc[(df['HomePlanet'].isnull()) & (df['Deck'] == 'G'), 'HomePlanet'] = 'Earth'
df.loc[(df['CryoSleep'].isnull()) & (df['TotalSpend'] > 0), 'CryoSleep'] = False
df.loc[(df['CryoSleep'].isnull()) & (df['TotalSpend'] == 0), 'CryoSleep'] = True

# 남은 결측치 최빈값/중앙값 처리
for col in ['HomePlanet', 'Destination', 'CryoSleep', 'VIP', 'Side', 'Deck']:
    df[col] = df[col].fillna(df[col].mode()[0])
    
for col in ['Age'] + spend_feats:
    df[col] = df[col].fillna(df[col].median())

# 4. 불필요한 노이즈 변수 제거 및 인코딩
df = df.drop(['PassengerId', 'Name', 'Cabin', 'Group'], axis=1)

# 범주형 변수 원-핫 인코딩
categorical = ['HomePlanet', 'Destination', 'Deck', 'Side', 'CryoSleep', 'VIP']
df = pd.get_dummies(df, columns=categorical, drop_first=True)

# 5. 데이터 재분리
X_train = df.iloc[:len(train)]
X_test = df.iloc[len(train):]

# 6. 강력한 두 모델의 소프트 보팅 앙상블 (Voting Ensemble)
hgb = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=6, 
    min_samples_leaf=15, l2_regularization=0.1, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_split=10, 
    min_samples_leaf=5, random_state=42
)

# 두 모델의 예측 확률을 평균 내서 최종 결정
model = VotingClassifier(estimators=[('hgb', hgb), ('rf', rf)], voting='soft')

print("🔥 모델 학습 시작...")
model.fit(X_train, y)

# 7. 예측 및 제출 파일 생성
preds = model.predict(X_test)
submission = pd.DataFrame({
    'PassengerId': passengers,
    'Transported': preds.astype(bool)
})

submission.to_csv('submission.csv', index=False)
print("✅ 최종 'submission.csv' 저장 완료!")

🔥 모델 학습 시작...
✅ 최종 'submission.csv' 저장 완료!


## 회고
제미나이 없었으면 하지도 못했을거다 아직은 한참 부족한데 계속 공부해서 제미나이 도움없이도 한번 해보고싶다 그때가지 계속 정진 !!


![Kaggle Score](2.png)